# Pareto Design Dashboard

## What you will learn
How to compare designs when no scalar objective is objectively final.

## Codes used
Pure Python cached metrics table.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
`11_pareto_front.png` and `11_weighted_selection.png`.


In [ ]:

from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")


In [ ]:

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Use cached mode first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")


In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:

import matplotlib.pyplot as plt
from sos2026.pareto import design_table, nondominated, weighted_selection
from sos2026.plotting import savefig, caption
df = design_table()
keep = nondominated(df, ["epsilon_eff", "coil_length", "turbulence_proxy"])
fig, ax = plt.subplots(figsize=(6.2, 3.9))
ax.scatter(df["coil_length"], df["epsilon_eff"], c=df["turbulence_proxy"], cmap="plasma", s=90, edgecolor="black")
ax.scatter(df.loc[keep,"coil_length"], df.loc[keep,"epsilon_eff"], facecolors="none", edgecolors="lime", s=180, linewidths=2)
for _, row in df.iterrows():
    ax.text(row["coil_length"]+0.01, row["epsilon_eff"], row["design"], fontsize=7)
ax.set_xlabel("coil length")
ax.set_ylabel("epsilon_eff")
ax.grid(alpha=0.25)
caption(ax, "Nondominated designs reveal tradeoffs a single objective can hide.")
savefig(fig, "11_pareto_front.png")
ranked = weighted_selection(df, {"epsilon_eff":0.35, "coil_length":0.25, "turbulence_proxy":0.25, "profile_error":0.15})
print(ranked[["design", "weighted_score"]].head())
print("Caption: changing weights is a design decision; record it explicitly.")


## Try this
Change the weights to favor coil simplicity and observe the selected design.

## Expected qualitative answer
The best weighted design shifts toward easier coils, often sacrificing transport metrics.

## Research extension
Replace one cached column with a real metric from another notebook and recompute the front.
